# GraphLens — Link Prediction Demo

This notebook walks through the full ML pipeline:
1. Load a KG JSON (or merge several)
2. Inspect graph statistics
3. Run the heuristic baseline
4. Train a TransE embedding model (requires `graphlens[ml]`)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))  # repo root

from graphlens.ml.graph_builder import KGGraph
from graphlens.ml.link_prediction import HeuristicPredictor, PyKEENPredictor

## 1 — Load and inspect the graph

In [ ]:
kg = KGGraph.from_json('../data/output/results.json')

# Merge multiple extractions:
# kg = KGGraph.merge([
#     KGGraph.from_json('../data/output/doc1.json'),
#     KGGraph.from_json('../data/output/doc2.json'),
# ])

import json
print(json.dumps(kg.stats(), indent=2))

## 2 — Visualise the graph (NetworkX + Matplotlib)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = kg.graph.to_undirected()

type_colors = {
    'PERSON': '#4e79a7', 'ORG': '#f28e2b', 'LOCATION': '#59a14f',
    'CONCEPT': '#e15759', 'PRODUCT': '#76b7b2', 'EVENT': '#edc948',
    'OTHER': '#b07aa1',
}
colors = [type_colors.get(G.nodes[n].get('type', 'OTHER'), '#aaa') for n in G.nodes]
labels = {n: n for n in G.nodes}

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(
    G, pos, labels=labels, node_color=colors,
    node_size=1200, font_size=9, arrows=False,
)
plt.title('Knowledge Graph')
plt.axis('off')
plt.tight_layout()
plt.show()

## 3 — Train / test split

In [ ]:
train_triples, test_triples = kg.train_test_split(test_size=0.2, seed=42)
print(f'Train: {len(train_triples)}  Test: {len(test_triples)}')

## 4 — Heuristic baseline (no ML deps)

In [ ]:
heuristic = HeuristicPredictor(kg, method='common_neighbors')
metrics   = heuristic.evaluate(test_triples, train_triples)
print(json.dumps(metrics, indent=2))

In [ ]:
# Predict top-5 missing links for an entity
entity = kg.id_to_entity[0]   # first entity in the graph
print(f'Top predictions for "{entity}":')
for tail, score in heuristic.predict_tails(entity, top_k=5):
    print(f'  {score:.3f}  {tail}')

## 5 — TransE embedding model

Requires: `pip install graphlens[ml]`

In [ ]:
predictor = PyKEENPredictor(kg, model_name='TransE', epochs=100, embedding_dim=64)
predictor.train(train_triples)

e_metrics = predictor.evaluate(test_triples)
print(json.dumps(e_metrics, indent=2))

In [ ]:
# Predict specific tails
entity   = kg.id_to_entity[0]
relation = kg.id_to_relation[0]
print(f'Top predictions for ({entity!r}, {relation!r}) → ?')
for tail, score in predictor.predict_tails(entity, relation, top_k=5):
    print(f'  {score:+.3f}  {tail}')